In [13]:
import pandas as pd
import numpy as np
import belo_horizonte_real_estate_market.functions.match_info as match_info
import belo_horizonte_real_estate_market.functions.sources as dicts

In [14]:
pd.set_option('display.max_columns', None)

In [15]:
df_enderecamento = pd.read_parquet(path = "../data/processed_enderecamento.parquet")

In [16]:
df_nulotctm_enderecos = df_enderecamento\
.query("nulotctm.notna()")\
.groupby(["nulotctm", "id_quadra_ctm", "area_m2", "codigo_zh"], as_index = False, dropna = False)\
.agg({"idend": list, "id_logradouro": list, "id_edc": list, "nome_bairro_popular": list})\
.assign(idend = lambda df: df["idend"].apply(lambda x: sorted(list(set(x)))))\
.assign(id_logradouro = lambda df: df["id_logradouro"].apply(lambda x: sorted(list(set(x)))))\
.assign(id_edc = lambda df: df["id_edc"].apply(lambda x: sorted(list(set(x)))))\
.assign(nome_bairro_popular = lambda df: df["nome_bairro_popular"].apply(lambda x: sorted(list(set(x)))))\
.rename(columns = {"idend": "list_idend", "id_logradouro": "list_id_logradouro", "id_edc": "list_id_edc", "nome_bairro_popular": "list_bairro"})

In [17]:
df_nulotctm_enderecos

,nulotctm,id_quadra_ctm,area_m2,codigo_zh,list_idend,list_id_logradouro,list_id_edc,list_bairro
0,100001400020,9025,240.44,oe120,[01711400020],[17114],[96563],[alto barroca]
1,100001400120,9025,130.94,oe120,"[01711400122, 01711400122A]",[17114],"[690480, 96516]",[alto barroca]
2,100001400130,9025,160.09,oe120,[01711400132],[17114],[96515],[alto barroca]
3,100001400170,9025,299.22,oe120,[01711400174],[17114],[96511],[alto barroca]
4,100001400180,9025,292.72,oe120,[01711400186],[17114],[96510],[alto barroca]
...,...,...,...,...,...,...,...,...
351411,91230600305,48991,302.29,cs408,[04481500342],[44815],[239684],[santo antonio]
351412,91230600315,48991,300.99,cs408,[04481500354],[44815],[239218],[santo antonio]
351413,91230600325,48991,290.39,cs408,[04481500364],[44815],[239216],[santo antonio]
351414,91231900015,48992,2265.37,cs408,"[05577400535, 05577400535A, 05577400549, 05577...","[55774, 57726, 67162]","[238016, 239693, 239698, 422804, 462013, 46201...",[santo antonio]


In [18]:
cols_cadastro_imobiliario = ["id_iptu_ctm", "nulotctm", "tipo_logradouro", "nome_logradouro", "numero_imovel", "cep", "indice_cadastral", "complemento_endereco", 
                             "ano_construcao", "zoneamento_pviptu", "area_terreno", "area_construcao", "tipo_construtivo", "tipo_ocupacao", "padrao_acabamento", 
                             "quantidade_economias", "fracao_ideal", "zona_homogenia", "tipologia", "geometria"]

df_cadastro_imobiliario_bhmap = pd.concat(objs = [pd.read_parquet(path = "../data/raw_cadastro_imobiliario_bhmap_split0.parquet"),
                                                  pd.read_parquet(path = "../data/raw_cadastro_imobiliario_bhmap_split1.parquet"),
                                                  pd.read_parquet(path = "../data/raw_cadastro_imobiliario_bhmap_split2.parquet")],
                                          ignore_index = True)\
.astype("str")\
.assign(numero_imovel = lambda df: df["numero_imovel"].fillna("0").str.replace("\.0", "", regex = True))\
.assign(ind_cep = lambda df: df["cep"].apply(lambda x: len(x)))\
.assign(cep = lambda df : np.where(df["ind_cep"] == 8, df["cep"].apply(lambda x: x[:5] + "-" + x[5:]), ""))\
.assign(complemento_endereco = lambda df: df["complemento_endereco"].fillna(""))\
.assign(tipologia = lambda df: df["tipologia"].fillna(""))\
.assign(area_construcao = lambda df: df["area_construcao"].fillna("0"))\
.astype(dtype = {"area_terreno": "float", "area_construcao": "float", "fracao_ideal": "float", "quantidade_economias": "int", "ano_construcao": "int"})\
[cols_cadastro_imobiliario]\
.assign(tipo_logradouro = lambda df: df["tipo_logradouro"].map(dicts.DATASETS["TIPO_LOGRADOURO"]))

In [19]:
df_cadastro_imobiliario = pd.read_parquet(path = "../data/raw_cadastro_imobiliario.parquet")\
.astype("str")\
.assign(numero_imovel = lambda df: df["numero_imovel"].fillna("0").str.replace("\.0", "", regex = True))\
.assign(ind_cep = lambda df: df["cep"].apply(lambda x: len(x)))\
.assign(cep = lambda df : np.where(df["ind_cep"] == 8, df["cep"].apply(lambda x: x[:5] + "-" + x[5:]), ""))\
.assign(tipologia = lambda df: df["tipologia"].fillna(""))\
.assign(area_construcao = lambda df: df["area_construcao"].fillna("0"))\
.assign(tipo_logradouro = lambda df: df["tipo_logradouro"].map(dicts.DATASETS["TIPO_LOGRADOURO"]))\
.astype(dtype = {"area_terreno": "float", "area_construcao": "float", "fracao_ideal": "float", "quantidade_economias": "float"})\
.astype(dtype = {"quantidade_economias": "int"})\
.assign(nome_regional = lambda df: df["urlfile"].str.extract(r"^(.+regional_)(.+?)(_cadastro.+)$")[1])\
.assign(nome_regional = lambda df: df["nome_regional"].str.replace("_", " ").str.upper())

In [20]:
df_processed_cadastro_imobiliario = pd.concat(objs = 
    [df_cadastro_imobiliario_bhmap.drop(columns = ["ano_construcao", "complemento_endereco"]),
     df_cadastro_imobiliario.drop(columns = ["nome_regional", "urlfile", "ind_cep"])])\
.drop_duplicates(["nulotctm", "indice_cadastral"])\
.merge(df_cadastro_imobiliario_bhmap[["nulotctm", "indice_cadastral", "ano_construcao", "complemento_endereco"]], how = "left")\
.merge(df_cadastro_imobiliario[["nulotctm", "indice_cadastral", "nome_regional"]], how = "left")\
.merge(df_nulotctm_enderecos, how = "left")


In [21]:
df_logradouros = pd.read_parquet(path = "../data/processed_logradouros.parquet")
df_logradouros

,id_logradouro,desc_tipo_logradouro,nome_logradouro,cep,bairro,nome_regional,codigo_zh,size,logradouro,cod1,cod2,cod3,prefixo_cep,sufixo_cep,ind_nome_unico,ind_cep_unico,ind_bairro,ind_regional,ind_zh,ind_cod1_unico,ind_cod2_unico,ind_cod3_unico,ind_prefixo_cep,ind_sufixo_cep,ind_cod1_regional,ind_cod2_regional,ind_cod3_regional,ind_cod1_zh,ind_cod2_zh,ind_cod3_zh,ind_cod1_bairro,ind_cod2_bairro,ind_cod3_bairro,ind_cod1_cep,ind_cod2_cep,ind_cod3_cep,ind_cod1_prefixo_cep,ind_cod2_prefixo_cep,ind_cod3_prefixo_cep,ind_cod1_sufixo_cep,ind_cod2_sufixo_cep,ind_cod3_sufixo_cep,num_bairros,num_regional,largura_media,comprimento_logradouro,soma_ind
index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
8719,28133,rua,fernao dias,30285-160,alto vera cruz,leste,le319,565,rua fernao dias,rfrnds,uaeaoia,fernaodias,30285,160,1,1,1,1,1,1,0,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,4,1,11.64,2856.4,26
7114,19917,rua,desembargador braulio,30285-170,alto vera cruz,leste,le319,547,rua desembargador braulio,rdsmbrgdrbrl,uaeeaaoauio,desembargadorbraulio,30285,170,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,3,1,9.7,2546.85,28
23411,66535,rua,desembargador saraiva,30285-150,alto vera cruz,leste,le319,501,rua desembargador saraiva,rdsmbrgdrsrv,uaeeaaoaaia,desembargadorsaraiva,30285,150,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,9.38,1522.05,28
25567,78241,rua,coletora,30670-050,vila pinho,barreiro,ba227,494,rua coletora,rcltr,uaoeoa,coletora,30670,050,1,1,1,1,1,1,0,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,1,13.24,1517.08,26
1576,10878,rua,padre argemiro moreira,31995-162,beira-linha,nordeste,ne412,485,rua padre argemiro moreira,rpdrrgmrmrr,uaaeaeiooeia,padreargemiromoreira,31995,162,1,1,1,1,1,1,1,0,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,14,2,10.46,10357.68,26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16646,36734,rua,jacui,31110-050,floresta,leste,le205,1,rua jacui,rjc,uaaui,jacui,31110,050,0,1,1,1,1,0,0,0,1,1,0,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,7,2,16.26,4359.85,22
16635,36690,rua,jacinto olau,31150-430,santa cruz,nordeste,ne110,1,rua jacinto olau,rjcntl,uaaiooau,jacintoolau,31150,430,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,9.98,285.55,28
6,100029,rua,euclides franco,31370-250,braunas,pampulha,pa107,1,rua euclides franco,rcldsfrnc,uaeuieao,euclidesfranco,31370,250,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,3,1,14.97,832.77,28


In [22]:
regex_tp_logradouro = "^(ruadepedestre|rua|avenida|alameda|beco|estrada|praca|rodovia|travessa|viadepedestre|viaduto|via|largo|espacolivredeusopublico|trincheira|acesso|tunel)"

df_logradouros_ci_bhmap = df_processed_cadastro_imobiliario\
.query("list_id_logradouro.isna()")\
.assign(codigo_zh = lambda df: df["zona_homogenia"].str.lower())\
[["nulotctm", "tipo_logradouro", "nome_logradouro", "cep", "nome_regional", "codigo_zh", "tipo_construtivo", "tipo_ocupacao"]]\
.drop_duplicates()\
.assign(logradouro = lambda df: df["tipo_logradouro"] + " " + df["nome_logradouro"].str.replace("\'", "", regex = True))\
.drop(columns = ["tipo_logradouro", "nome_logradouro"])\
.apply(lambda x: x.str.lower())\
.assign(cod1 = lambda df: df["logradouro"].str.replace("a|e|i|o|u|á|ã|é|ê|í|ó|õ|ô|ú|û|ç|\'|\s+", "", regex = True))\
.assign(cod2 = lambda df: df["logradouro"].str.replace("b|c|d|f|g|h|j|k|l|m|n|p|q|r|s|t|v|x|w|y|z|\s+", "", regex = True))\
.assign(cod3 = lambda df: df["logradouro"].str.replace("\'|\s+", "", regex = True))\
.assign(cod3 = lambda df: df["cod3"].str.replace(regex_tp_logradouro, "", regex = True))\
.assign(prefixo_cep = lambda df: df["cep"].apply(lambda x: x[:5]))\
.assign(sufixo_cep = lambda df: df["cep"].apply(lambda x: x[-3:]))\
.pipe(func = match_info.match_logradouro, df_logradouros = df_logradouros)\
.query("id_logradouro.notna()")\
.groupby("nulotctm", as_index = False, dropna = False)\
.agg({"id_logradouro": lambda x: sorted(list(set(x)))})

In [25]:
df_processed_cadastro_imobiliario = df_processed_cadastro_imobiliario\
.merge(df_logradouros_ci_bhmap, how = "left")\
.assign(list_id_logradouro = lambda df: df["list_id_logradouro"].fillna(df["id_logradouro"]))\
.reset_index(drop = True)\
.assign(split = lambda df: df.index % 5)\
[['id_quadra_ctm', 'nulotctm', 'indice_cadastral', 'tipo_logradouro', 'nome_logradouro', 'numero_imovel', 
  'cep', 'complemento_endereco', 'nome_regional', 'list_idend', 'list_id_logradouro', 'list_id_edc', 
  'list_bairro', 'ano_construcao', 'zoneamento_pviptu', 'area_terreno', 'area_m2', 'area_construcao', 
  'tipo_construtivo', 'tipo_ocupacao', 'padrao_acabamento', 'quantidade_economias', 'fracao_ideal', 
  'zona_homogenia', 'tipologia', 'geometria', "split"]]

In [27]:
df_processed_cadastro_imobiliario.query("split == 0").to_parquet(path = "../data/processed_cadastro_imobiliario_split0.parquet", engine = "fastparquet", compression = "zstd")
df_processed_cadastro_imobiliario.query("split == 1").to_parquet(path = "../data/processed_cadastro_imobiliario_split1.parquet", engine = "fastparquet", compression = "zstd")
df_processed_cadastro_imobiliario.query("split == 2").to_parquet(path = "../data/processed_cadastro_imobiliario_split2.parquet", engine = "fastparquet", compression = "zstd")
df_processed_cadastro_imobiliario.query("split == 3").to_parquet(path = "../data/processed_cadastro_imobiliario_split3.parquet", engine = "fastparquet", compression = "zstd")
df_processed_cadastro_imobiliario.query("split == 4").to_parquet(path = "../data/processed_cadastro_imobiliario_split4.parquet", engine = "fastparquet", compression = "zstd")
df_processed_cadastro_imobiliario

,id_quadra_ctm,nulotctm,indice_cadastral,tipo_logradouro,nome_logradouro,numero_imovel,cep,complemento_endereco,nome_regional,list_idend,list_id_logradouro,list_id_edc,list_bairro,ano_construcao,zoneamento_pviptu,area_terreno,area_m2,area_construcao,tipo_construtivo,tipo_ocupacao,padrao_acabamento,quantidade_economias,fracao_ideal,zona_homogenia,tipologia,geometria,split
0,1671,40225200480,354008 033A0017,RUA,PRIMEIRO DE MAIO,290,31130-130,None,NORDESTE,"[04889800039, 04889800290A, 05523900282, 05523...","[48898, 55239]","[28537, 28538, 28551, 334261, 696800, 858474]",[cachoeirinha],1960.0,ZAR2,319.00,348.09,283.00,CASA,RESIDENCIAL,P2,4,1.000000,NE116,DEMAIS CASOS,MULTIPOLYGON (((610000.247313316 7799916.18066...,0
1,534,10277400080,006048 302 0211,AVENIDA,AFONSO PENA,1715,30130-006,BLOCO A APT 1102,CENTRO SUL,"[00125901707, 00125901709, 00125901711, 001259...","[1259, 13081]","[378859, 433865, 439499, 441375, 443185, 44318...",[funcionarios],1968.0,ZCBH,954.00,971.09,120.25,APARTAMENTO,RESIDENCIAL,P3,1,0.010280,CS125,ALINHAMENTO,MULTIPOLYGON (((611670.533779752 7796193.77834...,1
2,329,10202400400,007033 011 1600,AVENIDA,DO CONTORNO,4480,30110-028,SALA 606,CENTRO SUL,"[01722804474, 01722804480, 01722804482]",[17228],"[375061, 453529, 829368]",[funcionarios],1994.0,ZCBH,1694.00,1653.58,48.13,SALA,NAO RESIDENCIAL,P4,1,0.005167,CS214,None,MULTIPOLYGON (((612527.96140718 7795524.254629...,2
3,90,10308300305,008015 019 0176,AVENIDA,AUGUSTO DE LIMA,1259,30190-002,LOJA 16A,CENTRO SUL,"[00673101255, 00673101255A, 00673101255B, 0067...",[6731],"[10173, 10174, 10175, 10176, 10177, 10189, 434...",[barro preto],1996.0,ZCBH,1200.00,1085.67,44.77,LOJA,NAO RESIDENCIAL,P4,1,0.009470,CS129,LOJA EM EDIFICIO/GALERIA-INFERIOR,MULTIPOLYGON (((610072.439680128 7796763.13245...,3
4,242,10223900045,009018 003 0224,RUA,BERNARDO GUIMARAES,2014,30140-087,APT 801,CENTRO SUL,"[00936402004, 00936402014]",[9364],"[4388, 4811]",[lourdes],1983.0,ZCBH,1200.00,1196.36,122.00,APARTAMENTO,RESIDENCIAL,P4,1,0.014573,CS203,DEMAIS CASOS,MULTIPOLYGON (((610633.676580444 7796152.98665...,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
917235,15692,190561800170,970065 009 0041,RUA,JOSE LOURIVAL DA SILVA,81,31650-590,NaN,VENDA NOVA,"[05607000081, 05607000083]",[56070],"[591493, 755108]",[serra verde],NaN,ZAR2,360.00,366.42,72.47,APARTAMENTO,RESIDENCIAL,P2,1,0.160080,VN128,FRENTE,"POLYGON ((608627.3 7810541,608632.94 7810552,6...",0
917236,15692,190561800170,970065 009 0033,RUA,JOSE LOURIVAL DA SILVA,81,31650-590,NaN,VENDA NOVA,"[05607000081, 05607000083]",[56070],"[591493, 755108]",[serra verde],NaN,ZAR2,360.00,366.42,62.35,APARTAMENTO,RESIDENCIAL,P2,1,0.137720,VN128,FRENTE,"POLYGON ((608627.3 7810541,608632.94 7810552,6...",1
917237,15692,190561800170,970065 009 0068,RUA,JOSE LOURIVAL DA SILVA,83,31650-590,NaN,VENDA NOVA,"[05607000081, 05607000083]",[56070],"[591493, 755108]",[serra verde],NaN,ZAR2,360.00,366.42,179.55,CASA,RESIDENCIAL,P3,1,0.396600,VN128,FRENTE,"POLYGON ((608627.3 7810541,608632.94 7810552,6...",2
917238,15692,190561800170,970065 009 0025,RUA,JOSE LOURIVAL DA SILVA,81,31650-590,NaN,VENDA NOVA,"[05607000081, 05607000083]",[56070],"[591493, 755108]",[serra verde],NaN,ZAR2,360.00,366.42,62.29,APARTAMENTO,RESIDENCIAL,P2,1,0.137590,VN128,FRENTE,"POLYGON ((608627.3 7810541,608632.94 7810552,6...",3
